In [1]:
__author__ = 'Cyrille Mvomo, https://github.com/cyrillemvomo'
__version__ = "1"
__license__ = "MIT"

**Import**

In [2]:
import pandas as pd
import numpy as np
import pickle


**Source Demographic, Home and Lab Gait Data**

- Demographic

In [3]:
# Read and Only keep columns of interest
DEMOGRAPHIC = pd.read_csv("/Volumes/Active/Datasets/A_Principal_Component_Model_Provides_Gait_Features_for_Tracking_Symptoms_Severity_and_Neural_Progres/MJFF_Cohort/Demographic.csv").sort_values(by= ["subject_id"])
DEMOGRAPHIC = DEMOGRAPHIC[["subject_id", 'height', "lower_limb_length", "gender", "birth_year",
                           "diagnosis_year", "pd_most_affected_side", 'gait_impediments',
                            'posture_instability', 'tremor', 'bradykinesia', 'disrupted_sleep',
                            'freeze_of_gait', 'dyskinesia', 'rigidity',
                           "updrs_score_p1", "updrs_score_p2", "updrs_score_p3", "updrs_score_p4",
                           "h_and_y_score", "updrs_second_visit_score_p3"]]

In [4]:
# Add time of end and start of each laboratory tasks of interest as columns
TASKS = pd.read_csv("/Volumes/Active/Datasets/A_Principal_Component_Model_Provides_Gait_Features_for_Tracking_Symptoms_Severity_and_Neural_Progres/MJFF_Cohort/Legend_Lab_Tasks.csv").sort_values(by= ["subject_id"])

STRAIGHT_ON = TASKS[["subject_id", "visit", "task_code", "repetition", "timestamp_start", "timestamp_end", "phenotype"]].loc[(TASKS["task_code"] == "wlkgs") & (TASKS["visit"] == 1)]
STRAIGHT_OFF = TASKS[["subject_id", "visit", "task_code", "repetition", "timestamp_start", "timestamp_end", "phenotype"]].loc[(TASKS["task_code"] == "wlkgs") & (TASKS["visit"] == 2)]

DT_ON = TASKS[["subject_id", "visit", "task_code", "repetition", "timestamp_start", "timestamp_end", "phenotype"]].loc[(TASKS["task_code"] == "wlkgc") & (TASKS["visit"] == 1)]
DT_OFF = TASKS[["subject_id", "visit", "task_code", "repetition", "timestamp_start", "timestamp_end", "phenotype"]].loc[(TASKS["task_code"] == "wlkgc") & (TASKS["visit"] == 2)]

STRAIGHT_ON_START, STRAIGHT_ON_STOP = [], []
STRAIGHT_OFF_START, STRAIGHT_OFF_STOP = [], []
DT_ON_START, DT_ON_STOP = [], []
DT_OFF_START, DT_OFF_STOP = [], []
for participant in DEMOGRAPHIC["subject_id"].values:
    DATA_STRAIGHT_ON = STRAIGHT_ON.loc[STRAIGHT_ON["subject_id"] == participant]
    DATA_STRAIGHT_OFF = STRAIGHT_OFF.loc[STRAIGHT_OFF["subject_id"] == participant]
    DATA_DT_ON = DT_ON.loc[DT_ON["subject_id"] == participant]
    DATA_DT_OFF = DT_OFF.loc[DT_OFF["subject_id"] == participant]
    #Add the second trial for each participant
    STRAIGHT_ON_START.append(DATA_STRAIGHT_ON.timestamp_start.values[1])
    STRAIGHT_ON_STOP.append(DATA_STRAIGHT_ON.timestamp_end.values[1])
    STRAIGHT_OFF_START.append(DATA_STRAIGHT_OFF.timestamp_start.values[1])
    STRAIGHT_OFF_STOP.append(DATA_STRAIGHT_OFF.timestamp_end.values[1])
    DT_ON_START.append(DATA_DT_ON.timestamp_start.values[1])
    DT_ON_STOP.append(DATA_DT_ON.timestamp_end.values[1])
    DT_OFF_START.append(DATA_DT_OFF.timestamp_start.values[1])
    DT_OFF_STOP.append(DATA_DT_OFF.timestamp_end.values[1])


# Add as columns
DEMOGRAPHIC["STRAIGHT_ON_START"] = STRAIGHT_ON_START
DEMOGRAPHIC["STRAIGHT_ON_STOP"] = STRAIGHT_ON_STOP
DEMOGRAPHIC["STRAIGHT_OFF_START"] = STRAIGHT_OFF_START
DEMOGRAPHIC["STRAIGHT_OFF_STOP"] = STRAIGHT_OFF_STOP
DEMOGRAPHIC["DT_ON_START"] = DT_ON_START
DEMOGRAPHIC["DT_ON_STOP"] = DT_ON_STOP
DEMOGRAPHIC["DT_OFF_START"] = DT_OFF_START
DEMOGRAPHIC["DT_OFF_STOP"] = DT_OFF_STOP

# Reset the indexes
DEMOGRAPHIC = DEMOGRAPHIC.reset_index(drop=True)

- Gait Data

In [20]:
#Function
def Get_Vertical_Acceleration(Dataset):
        ''' Function to determine vertical axis (i.e. column with mean acc closest to g) and randomly assign the two other columns as ML or AP

            Input: 
                - Dataset: DF of 4 columns like (timestamp, x, y, z)
            Output: 
                - COLUMNS_OF_INTEREST : List of 4 element like (timestamp, vertical, randomML, randomAP)
        '''
        g = 9.81
        mean_Column_1 = np.abs(g - np.mean(np.delete(np.abs(Dataset.iloc[:, 1].values), np.where(np.abs(Dataset.iloc[:, 1].values) * 0 !=0)[0])))
        mean_Column_2 = np.abs(g - np.mean(np.delete(np.abs(Dataset.iloc[:, 2].values), np.where(np.abs(Dataset.iloc[:, 2].values) * 0 !=0)[0])))
        mean_Column_3 = np.abs(g - np.mean(np.delete(np.abs(Dataset.iloc[:, 3].values), np.where(np.abs(Dataset.iloc[:, 3].values) * 0 !=0)[0])))
        array = np.array([mean_Column_1, mean_Column_2, mean_Column_3])
        if np.argmin(array) == 0:
            VERTICAL_ACC_COLUMN_LABEL = Dataset.columns[1]
            ML_ACC_COLUMN_LABEL = Dataset.columns[2] # Randomly
            AP_ACC_COLUMN_LABEL = Dataset.columns[3] # Randomly
            COLUMNS_OF_INTEREST = ["timestamp", VERTICAL_ACC_COLUMN_LABEL, ML_ACC_COLUMN_LABEL, AP_ACC_COLUMN_LABEL]

        if np.argmin(array) == 1:
            VERTICAL_ACC_COLUMN_LABEL = Dataset.columns[2]
            ML_ACC_COLUMN_LABEL = Dataset.columns[1] # Randomly
            AP_ACC_COLUMN_LABEL = Dataset.columns[3] # Randomly
            COLUMNS_OF_INTEREST = ["timestamp", VERTICAL_ACC_COLUMN_LABEL, ML_ACC_COLUMN_LABEL, AP_ACC_COLUMN_LABEL]

        if np.argmin(array) == 2:
            VERTICAL_ACC_COLUMN_LABEL = Dataset.columns[3]
            ML_ACC_COLUMN_LABEL = Dataset.columns[1] # Randomly
            AP_ACC_COLUMN_LABEL = Dataset.columns[2] # Randomly
            COLUMNS_OF_INTEREST = ["timestamp", VERTICAL_ACC_COLUMN_LABEL, ML_ACC_COLUMN_LABEL, AP_ACC_COLUMN_LABEL]
        
        return COLUMNS_OF_INTEREST

In [21]:
# ALL_GAIT_DATA_PATH = "/Volumes/Active/Datasets/A_Principal_Component_Model_Provides_Gait_Features_for_Tracking_Symptoms_Severity_and_Neural_Progres/MJFF_Cohort/"
# SUBJECT_ID = DEMOGRAPHIC["subject_id"].values[22:23]
# for participant_data in SUBJECT_ID:
#         # STORE GAIT DATA LAB
#         FULL_DATA_ON = pd.read_csv(f"{ALL_GAIT_DATA_PATH + participant_data}/rawData_Day1.txt", delimiter="\t").map(float)
#         FULL_DATA_OFF = pd.read_csv(f"{ALL_GAIT_DATA_PATH + participant_data}/rawData_Day4.txt", delimiter="\t").map(float)
#         COLUMNS_OF_INTEREST_ON = Get_Vertical_Acceleration(FULL_DATA_ON)
       



In [22]:
ALL_GAIT_DATA_PATH = "/Volumes/Active/Datasets/A_Principal_Component_Model_Provides_Gait_Features_for_Tracking_Symptoms_Severity_and_Neural_Progres/MJFF_Cohort/"
SUBJECT_ID = DEMOGRAPHIC["subject_id"].values
GAIT_DATA_STRAIGHT_ON, GAIT_DATA_STRAIGHT_OFF = [], []
GAIT_DATA_DT_ON, GAIT_DATA_DT_OFF = [], []
GAIT_DATA_HOME_DAY_1, GAIT_DATA_HOME_DAY_2 = [], []

for participant_data in SUBJECT_ID:
    try:
        # STORE GAIT DATA LAB
        FULL_DATA_ON = pd.read_csv(f"{ALL_GAIT_DATA_PATH + participant_data}/rawData_Day1.txt", delimiter="\t").map(float)
        FULL_DATA_OFF = pd.read_csv(f"{ALL_GAIT_DATA_PATH + participant_data}/rawData_Day4.txt", delimiter="\t").map(float)
  
            # Get start and stop for straight
        STRAIGHT_ON_START_TIME = DEMOGRAPHIC.loc[DEMOGRAPHIC["subject_id"] == participant_data]["STRAIGHT_ON_START"].values[0]
        STRAIGHT_ON_STOP_TIME = DEMOGRAPHIC.loc[DEMOGRAPHIC["subject_id"] == participant_data]["STRAIGHT_ON_STOP"].values[0]
        STRAIGHT_OFF_START_TIME = DEMOGRAPHIC.loc[DEMOGRAPHIC["subject_id"] == participant_data]["STRAIGHT_OFF_START"].values[0]
        STRAIGHT_OFF_STOP_TIME = DEMOGRAPHIC.loc[DEMOGRAPHIC["subject_id"] == participant_data]["STRAIGHT_OFF_STOP"].values[0]

            # Get start and stop for DT
        DT_ON_START_TIME = DEMOGRAPHIC.loc[DEMOGRAPHIC["subject_id"] == participant_data]["DT_ON_START"].values[0]
        DT_ON_STOP_TIME = DEMOGRAPHIC.loc[DEMOGRAPHIC["subject_id"] == participant_data]["DT_ON_STOP"].values[0]
        DT_OFF_START_TIME = DEMOGRAPHIC.loc[DEMOGRAPHIC["subject_id"] == participant_data]["DT_OFF_START"].values[0]
        DT_OFF_STOP_TIME = DEMOGRAPHIC.loc[DEMOGRAPHIC["subject_id"] == participant_data]["DT_OFF_STOP"].values[0]

            #Store
        DF_STRAIGHT_ON = FULL_DATA_ON.loc[(FULL_DATA_ON["timestamp"] >= STRAIGHT_ON_START_TIME) & (FULL_DATA_ON["timestamp"] <= STRAIGHT_ON_STOP_TIME)]
        DF_STRAIGHT_OFF = FULL_DATA_OFF.loc[(FULL_DATA_OFF["timestamp"] >= STRAIGHT_OFF_START_TIME) & (FULL_DATA_OFF["timestamp"] <= STRAIGHT_OFF_STOP_TIME)]
        DF_DT_ON = FULL_DATA_ON.loc[(FULL_DATA_ON["timestamp"] >= DT_ON_START_TIME) & (FULL_DATA_ON["timestamp"] <= DT_ON_STOP_TIME)]
        DF_DT_OFF = FULL_DATA_OFF.loc[(FULL_DATA_OFF["timestamp"] >= DT_OFF_START_TIME) & (FULL_DATA_OFF["timestamp"] <= DT_OFF_STOP_TIME)]
                #Get Vertical Axes
        COLUMNS_OF_INTEREST_DF_STRAIGHT_ON = Get_Vertical_Acceleration(DF_STRAIGHT_ON)
        COLUMNS_OF_INTEREST_DF_STRAIGHT_OFF = Get_Vertical_Acceleration(DF_STRAIGHT_OFF)
        COLUMNS_OF_INTEREST_DF_DT_ON = Get_Vertical_Acceleration(DF_DT_ON)
        COLUMNS_OF_INTEREST_DF_DT_OFF = Get_Vertical_Acceleration(DF_DT_OFF)
                #Append
        GAIT_DATA_STRAIGHT_ON.append(DF_STRAIGHT_ON[COLUMNS_OF_INTEREST_DF_STRAIGHT_ON].reset_index(drop=True))
        GAIT_DATA_STRAIGHT_OFF.append(DF_STRAIGHT_OFF[COLUMNS_OF_INTEREST_DF_STRAIGHT_OFF].reset_index(drop=True))
        GAIT_DATA_DT_ON.append(DF_DT_ON[COLUMNS_OF_INTEREST_DF_DT_ON].reset_index(drop=True))
        GAIT_DATA_DT_OFF.append(DF_DT_OFF[COLUMNS_OF_INTEREST_DF_DT_OFF].reset_index(drop=True))

        # STORE GAIT DATA HOME
        GAIT_DATA_HOME_DAY_1.append(pd.read_csv(f"{ALL_GAIT_DATA_PATH + participant_data}/rawData_Day2.txt", delimiter="\t", usecols=[0, 1, 2, 3]).map(float)) # Vertical to be determined after sequence detection
        GAIT_DATA_HOME_DAY_2.append(pd.read_csv(f"{ALL_GAIT_DATA_PATH + participant_data}/rawData_Day3.txt", delimiter="\t", usecols=[0, 1, 2, 3]).map(float))

    except Exception as e:
        GAIT_DATA_STRAIGHT_ON.append(np.nan)
        GAIT_DATA_STRAIGHT_OFF.append(np.nan)
        GAIT_DATA_DT_ON.append(np.nan)
        GAIT_DATA_DT_OFF.append(np.nan)
        GAIT_DATA_HOME_DAY_1.append(np.nan)
        GAIT_DATA_HOME_DAY_2.append(np.nan)
        print(participant_data, 'not working as: ', e, ' so NaN stored')


10_BOS not working as:  Error tokenizing data. C error: Calling read(nbytes) on source failed. Try engine='python'.  so NaN stored


- Combine Gait and demographic and save as bin file

In [5]:
# Save  data as csv

DEMOGRAPHIC.to_csv(f"/Volumes/Active/Datasets/A_Principal_Component_Model_Provides_Gait_Features_for_Tracking_Symptoms_Severity_and_Neural_Progres/MJFF_Cohort/RawData_Sourced/DEMOGRAPHIC/DEMOGRAPHIC.csv", index = False)


In [ ]:
# # Store gait data in an organized dictionnary with participant label as key and their data as value
# GAIT_DICTIONNARY = {}
# for participant in range(len(SUBJECT_ID)): #Create the dictionnary
#     GAIT_DICTIONNARY[SUBJECT_ID[participant]] = {
#                                                     "GAIT_DATA_STRAIGHT_ON":GAIT_DATA_STRAIGHT_ON[participant],
#                                                     "GAIT_DATA_STRAIGHT_OFF":GAIT_DATA_STRAIGHT_OFF[participant],
#                                                     "GAIT_DATA_DT_ON":GAIT_DATA_DT_ON[participant],
#                                                     "GAIT_DATA_DT_OFF":GAIT_DATA_DT_OFF[participant],
#                                                     "GAIT_DATA_HOME_DAY_1":GAIT_DATA_HOME_DAY_1[participant],
#                                                     "GAIT_DATA_HOME_DAY_2":GAIT_DATA_HOME_DAY_2[participant]
#                                                     }
# MEDICAL_DICTIONNARY = {"Medical_Record": DEMOGRAPHIC}#Create the dictionnary
# # Combine Demographic infos with Gait infos as a single dictionnary
# RawData_Sourced = MEDICAL_DICTIONNARY
# RawData_Sourced.update(GAIT_DICTIONNARY) # combine the two dictionnaries 

# # # Save all data DO IT LATER MAYBE
# # with open("/Users/cyrilleetude/Documents/RawData_Sourced.bin", "wb") as f: # Binarize the data and save it to the .bin file
# #         pickle.dump(RawData_Sourced, f)



for data in range(len(GAIT_DATA_STRAIGHT_ON)):
      try:
            GAIT_DATA_STRAIGHT_ON[data].to_csv(f"/Volumes/Active/Datasets/A_Principal_Component_Model_Provides_Gait_Features_for_Tracking_Symptoms_Severity_and_Neural_Progres/MJFF_Cohort/RawData_Sourced/STRAIGHT_ON/{SUBJECT_ID[data]}.csv", index = False)
      except Exception as e:
            print(SUBJECT_ID[data], 'not stored')  
            continue
for data in range(len(GAIT_DATA_STRAIGHT_OFF)):
      try:
            GAIT_DATA_STRAIGHT_OFF[data].to_csv(f"/Volumes/Active/Datasets/A_Principal_Component_Model_Provides_Gait_Features_for_Tracking_Symptoms_Severity_and_Neural_Progres/MJFF_Cohort/RawData_Sourced/STRAIGHT_OFF/{SUBJECT_ID[data]}.csv", index = False)
      except Exception as e:
            print(SUBJECT_ID[data], 'not stored')  
            continue
for data in range(len(GAIT_DATA_DT_ON)):
      try:
            GAIT_DATA_DT_ON[data].to_csv(f"/Volumes/Active/Datasets/A_Principal_Component_Model_Provides_Gait_Features_for_Tracking_Symptoms_Severity_and_Neural_Progres/MJFF_Cohort/RawData_Sourced/DT_ON/{SUBJECT_ID[data]}.csv", index = False)
      except Exception as e:
            print(SUBJECT_ID[data], 'not stored')  
            continue
for data in range(len(GAIT_DATA_DT_OFF)):
      try:
            GAIT_DATA_DT_OFF[data].to_csv(f"/Volumes/Active/Datasets/A_Principal_Component_Model_Provides_Gait_Features_for_Tracking_Symptoms_Severity_and_Neural_Progres/MJFF_Cohort/RawData_Sourced/DT_OFF/{SUBJECT_ID[data]}.csv", index = False)
      except Exception as e:
            print(SUBJECT_ID[data], 'not stored')  
            continue
for data in range(len(GAIT_DATA_HOME_DAY_1)):
      try:
            GAIT_DATA_HOME_DAY_1[data].to_csv(f"/Volumes/Active/Datasets/A_Principal_Component_Model_Provides_Gait_Features_for_Tracking_Symptoms_Severity_and_Neural_Progres/MJFF_Cohort/RawData_Sourced/HomeDay1/{SUBJECT_ID[data]}.csv", index = False)
      except Exception as e:
            print(SUBJECT_ID[data], 'not stored')  
            continue
for data in range(len(GAIT_DATA_HOME_DAY_2)):
      try:
            GAIT_DATA_HOME_DAY_2[data].to_csv(f"/Volumes/Active/Datasets/A_Principal_Component_Model_Provides_Gait_Features_for_Tracking_Symptoms_Severity_and_Neural_Progres/MJFF_Cohort/RawData_Sourced/HomeDay2/{SUBJECT_ID[data]}.csv", index = False)
      except Exception as e:
            print(SUBJECT_ID[data], 'not stored')
            continue  


RERUN ALL TOMORROW (FS = 50HZ)

Then do (BEFORE 14h10):
- Sequence detection (home only)
- Sequence selection (home only)
- Signal Processing
- Event
- Step length
- Gait outcomes
- Final data

- THEN MERGE ALL DATASET and do PCA on lab

- Then Stat on the full data (single script R)
